## Pengecekan dan Pembersihan JSONL Berdasarkan Kata Kunci

In [ ]:
import json
import glob
from collections import Counter
import pandas as pd

# 1. Tentukan list keyword yang ingin dicek (tambahkan jika ada variasi lain)
keywords = [
    "cek kesehatan gratis",
    "ckg",
    "pemeriksaan kesehatan gratis",
    "pengobatan gratis",
    "berobat gratis"
]

jsonl_files = glob.glob("opini_cek_kesehatan_gratis.jsonl")

total_data = 0
data_with_keywords = 0
data_less_than_5_words = 0  # Konter baru untuk kalimat < 5 kata
data_without_keywords = []
keyword_match_counts = Counter()

print("Mulai Pengecekan Data JSONL...\n")

for file_path in jsonl_files:
    # Membuat nama file output, contoh: opini_cek_kesehatan_gratis_cleaned.jsonl
    output_file = file_path.replace('.jsonl', '_cleaned.jsonl')

    print(f"Membaca file: {file_path}")
    print(f"Menyiapkan file penyimpanan baru: {output_file}")

    # Buka file input untuk dibaca dan file output untuk ditulis
    with open(file_path, 'r', encoding='utf-8') as f, \
            open(output_file, 'w', encoding='utf-8') as f_out:

        for line_num, line in enumerate(f, 1):
            try:
                data = json.loads(line.strip())
                total_data += 1

                # Mengambil teks dari key "text", jadikan huruf kecil (lowercase) untuk dicocokkan
                text = str(data.get("text", "")).lower()

                # 2. Pengecekan jumlah kata (skip jika kurang dari 5 kata)
                if len(text.split()) < 5:
                    data_less_than_5_words += 1
                    continue

                # Tulis data yang lolos filter panjang kata ke file baru
                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')

                # Cek apakah setidaknya 1 keyword ada di text
                matched_keywords = [kw for kw in keywords if kw in text]

                if matched_keywords:
                    data_with_keywords += 1
                    # Mencatat setiap keyword yang cocok untuk statistik
                    for kw in matched_keywords:
                        keyword_match_counts[kw] += 1
                else:
                    # Simpan informasi data yang tidak memiliki satupun keyword
                    data_without_keywords.append({
                        "file": file_path,
                        "line": line_num,
                        # Berdasarkan kolom index jika ada
                        "id": data.get("Unnamed: 0", "Unknown"),
                        "text_snippet": text[:50] + "..."  # Cuplikan teks
                    })

            except json.JSONDecodeError:
                print(
                    f"  [!] Error parsing JSON di file {file_path} baris {line_num}")

# 3. Cetak Hasil/Summary
print("\n" + "="*40)
print("HASIL PENGECEKAN KESELURUHAN")
print("="*40)
print(f"Total file diproses      : {len(jsonl_files)}")
print(f"Total baris data         : {total_data}")
print(f"Data < 5 Kata (Dihapus)  : {data_less_than_5_words}")
print(f"Data Tersimpan (Cleaned) : {total_data - data_less_than_5_words}")
print(f"Data MENGANDUNG keyword  : {data_with_keywords}")
print(f"Data TANPA keyword       : {len(data_without_keywords)}")

print("\nDetail frekuensi keyword yang ditemukan:")
for kw, count in keyword_match_counts.items():
    print(f" - '{kw}': {count} data")

# 4. Kalau ternyata ada data tanpa keyword, Anda bisa mengecek cuplikannya
if data_without_keywords:
    print("\n" + "-"*40)
    print("Contoh 5 Data yang TIDAK MENGANDUNG keyword sama sekali:")
    # make a temp df for displaying the data without keywords
    df = pd.DataFrame(data_without_keywords)

    for item in data_without_keywords[:5]:
        print(
            f"File: {item['file']} | Baris: {item['line']} | Index: {item['id']}")
        print(f"Teks: {item['text_snippet']}\n")

if 'df' in locals():
    display(df.head())
    
df.head()

Mulai Pengecekan Data JSONL...

Membaca file: opini_cek_kesehatan_gratis.jsonl
Menyiapkan file penyimpanan baru: opini_cek_kesehatan_gratis_cleaned.jsonl

HASIL PENGECEKAN KESELURUHAN
Total file diproses      : 1
Total baris data         : 544
Data < 5 Kata (Dihapus)  : 18
Data Tersimpan (Cleaned) : 526
Data MENGANDUNG keyword  : 519
Data TANPA keyword       : 7

Detail frekuensi keyword yang ditemukan:
 - 'cek kesehatan gratis': 397 data
 - 'ckg': 328 data
 - 'pemeriksaan kesehatan gratis': 18 data
 - 'pengobatan gratis': 3 data

----------------------------------------
Contoh 5 Data yang TIDAK MENGANDUNG keyword sama sekali:
File: opini_cek_kesehatan_gratis.jsonl | Baris: 277 | Index: Unknown
Teks: dengan presiden prabowo, nias selatan dapat perhat...

File: opini_cek_kesehatan_gratis.jsonl | Baris: 499 | Index: Unknown
Teks: harapan untuk generasi sehat kian dekat berkat kom...

File: opini_cek_kesehatan_gratis.jsonl | Baris: 501 | Index: Unknown
Teks: ruang gerak program kesehatan 

,file,line,id,text_snippet
0,opini_cek_kesehatan_gratis.jsonl,277,Unknown,"dengan presiden prabowo, nias selatan dapat pe..."
1,opini_cek_kesehatan_gratis.jsonl,499,Unknown,harapan untuk generasi sehat kian dekat berkat...
2,opini_cek_kesehatan_gratis.jsonl,501,Unknown,ruang gerak program kesehatan anak kini semaki...
3,opini_cek_kesehatan_gratis.jsonl,506,Unknown,solusi nyata dari presiden prabowo lewat cek k...
4,opini_cek_kesehatan_gratis.jsonl,519,Unknown,prabowo dorong akses kesehatan mudah supaya ra...


## Delete Sentence that 5 Words or Less


In [ ]:

import json
import re

input_file = "2026-04-09T07-45_export.jsonl"
output_file = "2026-04-09T07-45_export_cleaned.jsonl"


def clean_text(text):
    if not isinstance(text, str):
        return text

    # Remove Markdown bold formatting (**text**)
    text = text.replace("**", "")

    # Remove HTML bold tags (<b> or <strong>) in case they exist
    text = re.sub(r'</?(b|strong)>', '', text, flags=re.IGNORECASE)

    # Remove leading and trailing whitespace
    return text.strip()


processed_count = 0

print(f"Membaca dari {input_file}...")

with open(input_file, 'r', encoding='utf-8') as infile, \
        open(output_file, 'w', encoding='utf-8') as outfile:

    for line_num, line in enumerate(infile, 1):
        if not line.strip():
            continue

        try:
            data = json.loads(line.strip())

            # Reformat all string values in the JSON object
            for key, value in data.items():
                if isinstance(value, str):
                    data[key] = clean_text(value)

            # Save it back as JSONL
            outfile.write(json.dumps(data, ensure_ascii=False) + '\n')
            processed_count += 1

        except json.JSONDecodeError:
            print(f"  [!] Error parsing JSON di baris {line_num}")

print(
    f"Selesai! {processed_count} data telah dibersihkan dan disimpan sebagai {output_file}")

Membaca dari 2026-04-09T07-45_export.jsonl...
Selesai! 517 data telah dibersihkan dan disimpan sebagai 2026-04-09T07-45_export_cleaned.jsonl


## Pengubahan kolom "narasi" menjadi "text" dan menyimpan hasilnya ke file JSONL baru

In [3]:
import json

input_file = "dataset_sentimen_analisis_program_CKG.jsonl"
output_file = "dataset_sentimen_analisis_program_CKG_revisi.jsonl"

processed_count = 0

print(f"Membaca dari {input_file}...")

with open(input_file, 'r', encoding='utf-8') as infile, \
        open(output_file, 'w', encoding='utf-8') as outfile:

    for line_num, line in enumerate(infile, 1):
        if not line.strip():
            continue

        try:
            data = json.loads(line.strip())

            # Cek apakah ada key "narasi", jika ada ubah menjadi "text"
            if "narasi" in data:
                data["text"] = data.pop("narasi")

            # Simpan kembali sebagai JSONL
            outfile.write(json.dumps(data, ensure_ascii=False) + '\n')
            processed_count += 1

        except json.JSONDecodeError:
            print(f"  [!] Error parsing JSON di baris {line_num}")

print(
    f"Selesai! {processed_count} data telah diproses dan disimpan sebagai {output_file}")

Membaca dari dataset_sentimen_analisis_program_CKG.jsonl...
Selesai! 517 data telah diproses dan disimpan sebagai dataset_sentimen_analisis_program_CKG_revisi.jsonl
